# Security Analysis: Quantum Channel Eavesdropping (Eve)

This notebook demonstrates how quantum mechanics guarantees detection of eavesdroppers, and sweeps interception rates to visualize error increases.

## 1. The Intercept-Resend Attack
An eavesdropper (Eve) intercepts qubits traveling from Alice to Bob, measures them, and resends reconstructed qubits to Bob. 
Because Eve does not know Alice's basis choice, she must guess. When she guesses incorrectly, she collapses the quantum state in a mismatched basis. This perturbation introduces an error rate of 25% under full eavesdropping, which Bob and Alice detect as QBER.

## 2. Running the Security Analysis Sweep

Let's sweep interception probabilities from $0.0$ to $1.0$ and calculate QBER trends.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Ensure QST package is in PYTHONPATH
sys.path.insert(0, str(Path.cwd().parent / "src"))

from qst.models.config import SimulationConfig, ProtocolType
from qst.orchestration.orchestrator import SimulationOrchestrator

probabilities = [0.0, 0.20, 0.40, 0.60, 0.80, 1.0]
avg_qbers = []
statuses = []

orchestrator = SimulationOrchestrator()

for p in probabilities:
    config = SimulationConfig(
        n_qubits=25,
        seed=42,
        interception_probability=p,
        repetitions=5,
        protocol=ProtocolType.BB84
    )
    res = orchestrator.run_many(config)
    avg_qbers.append(res.average_qber)
    statuses.append(res.simulations[0].security_metrics.status.value)
    print(f"Intercept: {p:<4.2f} | Avg QBER: {res.average_qber:<6.4f} | Status: {statuses[-1]}")

# Plotting results
plt.figure(figsize=(8, 5))
plt.plot(probabilities, avg_qbers, marker='o', linestyle='-', color='#007acc', label='Simulated QBER')
plt.axhline(y=0.25, color='r', linestyle='--', label='Theoretical Max (25%)')
plt.xlabel('Interception Probability')
plt.ylabel('QBER')
plt.title('Quantum Bit Error Rate (QBER) vs. Eavesdropper Interception Rate')
plt.grid(True)
plt.legend()
plt.show()